# 01 Apply tags (job task 1 of 2)

Job version of `04_Governance/tags/*` (README: "Default + Override" taxonomy). One task applies all three levels:

| Level | Tags | Scope |
|---|---|---|
| schema | `layer` | bronze, silver, gold |
| table | `layer`, `mimic_module` (+ `type`, `domain` in Gold) | every table in the three layers; Gold uses the config below (24 modelled tables plus the 3 POE tables) |
| column | `hipaa_identifiability`, `hl7_confidentiality` | every Gold column: explicit override, else the name-based identifier rule, else default `de_identified` / `N_Normal` |

Idempotent: `SET TAGS` overwrites the same key with the same value. Tags set by teammates under other keys
(`pii_level`, `access_tier`, `data_domain`) are left alone. Views are tagged `type=view`.

**Governed tags.** `type` and `domain` carry a tag policy on the catalog: only the values
listed in the policy are accepted (`UC_TAG_POLICY_VALUE_NOT_ALLOWED` otherwise, and the allowed
list is at the end of that message). Because a table's `layer`, `type` and `mimic_module` go
out in ONE `SET TAGS`, a rejected `type` also loses `layer` and `mimic_module` for that table.
Add a new domain (or `table` for Bronze/Silver) to the governed tag before adding it here - see
`04_Governance/tags/README.md`.

Widgets: `mode` = `dry_run` | `apply`; `layers` = comma list of schemas to tag (default `bronze,silver,gold`).

In [0]:
dbutils.widgets.dropdown("mode", "dry_run", ["dry_run", "apply"])
dbutils.widgets.text("catalog", "cdr_mimiciv")
dbutils.widgets.text("layers", "bronze,silver,gold")

MODE = dbutils.widgets.get("mode")
CATALOG = dbutils.widgets.get("catalog")
LAYERS = [l.strip() for l in dbutils.widgets.get("layers").split(",") if l.strip()]
assert CATALOG == "cdr_mimiciv"
assert set(LAYERS) <= {"bronze", "silver", "gold"}, LAYERS
print(f"mode={MODE} layers={LAYERS}")

mode=apply layers=['bronze', 'silver', 'gold']


In [0]:
# Gold table-level config: the 24 tables from tags/Gold_Table_level.py plus the 3 POE tables from the Gold design (§3.3, §6)
GOLD_TABLE_TAGS = {
    "dim_patient":                   {"type": "dimension", "domain": "demographics",   "mimic_module": "hosp"},
    "dim_location":                  {"type": "dimension", "domain": "encounter",      "mimic_module": "hosp"},
    "dim_admission_type":            {"type": "dimension", "domain": "encounter",      "mimic_module": "hosp"},
    "dim_provider":                  {"type": "dimension", "domain": "encounter",      "mimic_module": "hosp"},
    "dim_service":                   {"type": "dimension", "domain": "encounter",      "mimic_module": "hosp"},
    "dim_date":                      {"type": "dimension", "domain": "encounter",      "mimic_module": "hosp"},
    "fact_admission":                {"type": "fact",      "domain": "encounter",      "mimic_module": "hosp"},
    "fact_transfer":                 {"type": "fact",      "domain": "encounter",      "mimic_module": "hosp"},
    "fact_service":                  {"type": "fact",      "domain": "encounter",      "mimic_module": "hosp"},
    "dim_careunit":                  {"type": "dimension", "domain": "critical_care",  "mimic_module": "icu"},
    "dim_icu_item":                  {"type": "dimension", "domain": "critical_care",  "mimic_module": "icu"},
    "fact_icu_stay":                 {"type": "fact",      "domain": "critical_care",  "mimic_module": "icu"},
    "fact_icu_observation":          {"type": "fact",      "domain": "critical_care",  "mimic_module": "icu"},
    "dim_medication":                {"type": "dimension", "domain": "pharmacy",       "mimic_module": "hosp"},
    "dim_medication_route":          {"type": "dimension", "domain": "pharmacy",       "mimic_module": "hosp"},
    "fact_prescription":             {"type": "fact",      "domain": "pharmacy",       "mimic_module": "hosp"},
    "fact_medication_administration":{"type": "fact",      "domain": "pharmacy",       "mimic_module": "hosp"},
    "dim_lab_item":                  {"type": "dimension", "domain": "diagnostics",    "mimic_module": "hosp"},
    "dim_microbiology_test":         {"type": "dimension", "domain": "diagnostics",    "mimic_module": "hosp"},
    "fact_lab":                      {"type": "fact",      "domain": "diagnostics",    "mimic_module": "hosp"},
    "fact_microbiology":             {"type": "fact",      "domain": "diagnostics",    "mimic_module": "hosp"},
    "dim_diagnosis":                 {"type": "dimension", "domain": "clinical_coding","mimic_module": "hosp"},
    "dim_procedure":                 {"type": "dimension", "domain": "clinical_coding","mimic_module": "hosp"},
    "fact_diagnosis":                {"type": "fact",      "domain": "clinical_coding","mimic_module": "hosp"},
    "fact_procedure":                {"type": "fact",      "domain": "clinical_coding","mimic_module": "hosp"},
    # POE (new in the Gold rewrite)
    "dim_order_type":                {"type": "dimension", "domain": "orders",         "mimic_module": "hosp"},
    "fact_order":                    {"type": "fact",      "domain": "orders",         "mimic_module": "hosp"},
    "fact_order_detail":             {"type": "fact",      "domain": "orders",         "mimic_module": "hosp"},
}

# Bronze/Silver: mimic_module by source table name (MIMIC-IV docs: icu module tables, everything else hosp)
ICU_TABLES = {"icustays", "chartevents", "d_items", "datetimeevents", "inputevents", "outputevents",
              "procedureevents", "ingredientevents", "caregiver"}
def mimic_module(table): return "icu" if table.lower() in ICU_TABLES else "hosp"

# Gold column-level overrides (tags/Gold_Column_level.py plus identifiers and POE free text)
Q, R, V = "phi_quasi_identifier", "R_Restricted", "V_Very_Restricted"
COLUMN_OVERRIDES = {
    "dim_patient": {
        "subject_id":        {"hipaa_identifiability": "phi_direct_identifier", "hl7_confidentiality": R},
        "gender":            {"hipaa_identifiability": Q, "hl7_confidentiality": R},
        "anchor_age":        {"hipaa_identifiability": Q, "hl7_confidentiality": R},
        "anchor_year":       {"hipaa_identifiability": Q, "hl7_confidentiality": R},
        "anchor_year_group": {"hipaa_identifiability": Q, "hl7_confidentiality": R},
    },
    "dim_provider": {
        "provider_id":       {"hipaa_identifiability": "phi_direct_identifier", "hl7_confidentiality": R},
    },
    "fact_admission": {
        "insurance":         {"hipaa_identifiability": Q, "hl7_confidentiality": R},
        "language":          {"hipaa_identifiability": Q, "hl7_confidentiality": R},
        "marital_status":    {"hipaa_identifiability": Q, "hl7_confidentiality": R},
        "race":              {"hipaa_identifiability": Q, "hl7_confidentiality": R},
        "deathtime":         {"hipaa_identifiability": Q, "hl7_confidentiality": V},
        "age_at_admission":  {"hipaa_identifiability": Q, "hl7_confidentiality": R},   # derived from anchor_age
    },
    "fact_diagnosis": {
        "primary_diagnosis_flag": {"hipaa_identifiability": "de_identified", "hl7_confidentiality": R},
    },
    "fact_order_detail": {
        "field_value":       {"hipaa_identifiability": Q, "hl7_confidentiality": R},   # free text
    },
}
COLUMN_DEFAULT = {"hipaa_identifiability": "de_identified", "hl7_confidentiality": "N_Normal"}

# Name-based classification, applied when a column has no explicit override above. Without this, every
# identifier column that the override map does not happen to mention fell through to COLUMN_DEFAULT and was
# tagged `de_identified` / `N_Normal` - i.e. an unmasked identifier in any table that exposes raw ids
# (facts did, before patient_key/provider_key replaced them). Classification must not depend on someone
# remembering to list a column.
# Person and provider identifiers only. Row-level ids (transfer_id, labevent_id, emar_id, microevent_id,
# pharmacy_id, orderid) are deliberately NOT here: they identify a record, not a patient, and tagging them
# direct would make 05_fine_grained_controls mask them for every researcher with no privacy benefit.
DIRECT_IDENTIFIER_COLUMNS = {"subject_id", "provider_id", "caregiver_id"}
QUASI_IDENTIFIER_COLUMNS = {"hadm_id", "stay_id", "anchor_age", "anchor_year", "anchor_year_group"}

def column_tags(table, column):
    """Explicit override first, then the name-based rule, then the default."""
    override = COLUMN_OVERRIDES.get(table, {}).get(column)
    if override:
        return override
    name = column.lower()
    if name in DIRECT_IDENTIFIER_COLUMNS:
        return {"hipaa_identifiability": "phi_direct_identifier", "hl7_confidentiality": R}
    if name in QUASI_IDENTIFIER_COLUMNS:
        return {"hipaa_identifiability": Q, "hl7_confidentiality": R}
    return dict(COLUMN_DEFAULT)

In [0]:
def tag_clause(tags): return ", ".join(f"'{k}'='{v}'" for k, v in tags.items())

statements = []   # (label, sql)
for layer in LAYERS:
    statements.append((f"schema {layer}", f"ALTER SCHEMA {CATALOG}.{layer} SET TAGS ('layer'='{layer}')"))

tables = spark.sql(f"""
  SELECT table_schema, table_name, table_type FROM {CATALOG}.information_schema.tables
  WHERE table_schema IN ({", ".join(f"'{l}'" for l in LAYERS)}) ORDER BY 1, 2""").collect()

for t in tables:
    is_view = t.table_type == "VIEW"
    fqn = f"{CATALOG}.{t.table_schema}.{t.table_name}"
    if t.table_schema == "gold":
        cfg = GOLD_TABLE_TAGS.get(t.table_name)
        if cfg is None:
            print(f"WARNING: gold.{t.table_name} is not in GOLD_TABLE_TAGS; tagging layer/mimic_module only")
            cfg = {"type": "view" if is_view else "unclassified", "mimic_module": "hosp"}
        tags = {"layer": "gold", **cfg}
    else:
        tags = {"layer": t.table_schema, "type": "view" if is_view else "table", "mimic_module": mimic_module(t.table_name)}
    obj = "VIEW" if is_view else "TABLE"
    statements.append((f"{obj.lower()} {fqn}", f"ALTER {obj} {fqn} SET TAGS ({tag_clause(tags)})"))

missing = sorted(set(GOLD_TABLE_TAGS) - {t.table_name for t in tables if t.table_schema == "gold"})
if "gold" in LAYERS and missing:
    print("NOTE: configured Gold tables not present yet (build them with 03_Gold/99_Gold_Run_All):", missing)

# Column level. Views are skipped; a tag on the table carries no further.
#
# Gold: every base-table column, default plus override plus the name-based rule, so that an
# untagged column cannot exist there.
if "gold" in LAYERS:
    cols = spark.sql(f"""
      SELECT c.table_name, c.column_name FROM {CATALOG}.information_schema.columns c
      JOIN {CATALOG}.information_schema.tables t ON c.table_schema = t.table_schema AND c.table_name = t.table_name
      WHERE c.table_schema = 'gold' AND t.table_type <> 'VIEW' ORDER BY 1, c.ordinal_position""").collect()
    for c in cols:
        tags = column_tags(c.table_name, c.column_name)
        statements.append((f"column gold.{c.table_name}.{c.column_name}",
                           f"ALTER TABLE {CATALOG}.gold.{c.table_name} ALTER COLUMN `{c.column_name}` SET TAGS ({tag_clause(tags)})"))
    # overrides that point at columns that do not exist are a config error worth seeing
    have = {(c.table_name, c.column_name) for c in cols}
    for tbl, cmap in COLUMN_OVERRIDES.items():
        for col in cmap:
            if (tbl, col) not in have:
                print(f"WARNING: override for gold.{tbl}.{col} but column not found")

# Silver: the identifier columns only.
#
# Deliberately not every Silver column. Everything else in Silver classifies as the default
# (de_identified / N_Normal), which maps to no posture and therefore to no mask, so tagging ~600
# columns would add statements without changing a single policy. The identifier names are the
# ones already defined above, so the two layers cannot drift apart.
#
# Why this exists at all: 03_create_groups grants schema access from this same catalog, and a role
# whose `policy.direct_identifier` is `masked` must not be able to read the raw identifier one
# schema down. Applying the mask is the second layer behind the grant, not a substitute for it.
if "silver" in LAYERS:
    SILVER_ONLY_IDENTIFIERS = {"poe_id"}   # <subject_id>-<sequence> in MIMIC-IV: derived, not listed above
    silver_cols = spark.sql(f"""
      SELECT c.table_name, c.column_name FROM {CATALOG}.information_schema.columns c
      JOIN {CATALOG}.information_schema.tables t ON c.table_schema = t.table_schema AND c.table_name = t.table_name
      WHERE c.table_schema = 'silver' AND t.table_type <> 'VIEW' ORDER BY 1, c.ordinal_position""").collect()
    silver_tagged = 0
    for c in silver_cols:
        name = c.column_name.lower()
        if name in DIRECT_IDENTIFIER_COLUMNS or name in SILVER_ONLY_IDENTIFIERS:
            tags = {"hipaa_identifiability": "phi_direct_identifier", "hl7_confidentiality": R}
        elif name in QUASI_IDENTIFIER_COLUMNS:
            tags = {"hipaa_identifiability": Q, "hl7_confidentiality": R}
        else:
            continue
        silver_tagged += 1
        statements.append((f"column silver.{c.table_name}.{c.column_name}",
                           f"ALTER TABLE {CATALOG}.silver.{c.table_name} ALTER COLUMN `{c.column_name}` SET TAGS ({tag_clause(tags)})"))
    print(f"silver: {silver_tagged} identifier columns tagged "
          f"({sorted(DIRECT_IDENTIFIER_COLUMNS | QUASI_IDENTIFIER_COLUMNS | SILVER_ONLY_IDENTIFIERS)})")

print(f"{len(statements)} tag statements prepared")

silver: 67 identifier columns tagged (['anchor_age', 'anchor_year', 'anchor_year_group', 'caregiver_id', 'hadm_id', 'poe_id', 'provider_id', 'stay_id', 'subject_id'])
403 tag statements prepared


In [0]:
ok, errors = 0, []
for label, sql in statements:
    if MODE == "dry_run":
        print(sql)
        continue
    try:
        spark.sql(sql); ok += 1
    except Exception as e:
        # Keep the whole first line: for TAG_POLICY_VALUE_NOT_ALLOWED the list of values the
        # governed tag accepts sits at the END of the message, so a 200-character cut hides
        # exactly the information needed to fix the taxonomy.
        errors.append((label, str(e).splitlines()[0]))

print(f"\nmode={MODE} applied={ok} errors={len(errors)}")

# One line per distinct error. 66 rejections of three tag values are three problems, not 66.
by_message = {}
for label, message in errors:
    by_message.setdefault(message, []).append(label)
for message, labels in by_message.items():
    print(f"\nERROR ({len(labels)} statement(s)): {message}")
    print("    e.g.", ", ".join(labels[:4]) + (" ..." if len(labels) > 4 else ""))


mode=apply applied=403 errors=0


In [0]:
if MODE == "apply":
    display(spark.sql(f"""SELECT schema_name, tag_name, tag_value, COUNT(*) AS tables
                          FROM {CATALOG}.information_schema.table_tags
                          WHERE tag_name IN ('layer','type','domain','mimic_module') GROUP BY ALL ORDER BY 1,2,3"""))
    display(spark.sql(f"""SELECT tag_name, tag_value, COUNT(*) AS columns
                          FROM {CATALOG}.information_schema.column_tags
                          WHERE schema_name = 'gold' AND tag_name IN ('hipaa_identifiability','hl7_confidentiality')
                          GROUP BY ALL ORDER BY 1,2"""))
    assert not errors, f"{len(errors)} tag statements failed; see output"

dbutils.notebook.exit(f"01_apply_tags mode={MODE} statements={len(statements)} errors={len(errors)}")